# Part1: Market Setup


In [15]:
import random
import itertools

# -----------------------------
# Basic market objects
# -----------------------------
students = [f"i{n}" for n in range(1, 19)]
schools  = [f"s{n}" for n in range(1, 4)]
cap0     = {s: 6 for s in schools}  # capacity: 6 each

def random_prefs():
    """Strict random preferences for each student over schools."""
    return {i: random.sample(schools, len(schools)) for i in students}

def random_priorities():
    """Strict random priority ordering for each school over students."""
    return {s: random.sample(students, len(students)) for s in schools}

def rank_map(prefs):
    """rank_map[i][s] = rank position of school s in student i's preference list."""
    return {i: {sch: r+1 for r, sch in enumerate(lst)} for i, lst in prefs.items()}



# Part2: Matching Mechanism

### DA: student-proposing deferred acceptance

In [17]:
def deferred_acceptance(prefs, priorities, cap):
    pr_rank = {s: {stu: idx for idx, stu in enumerate(priorities[s])} for s in schools}
    held = {s: [] for s in schools}
    next_choice = {i: 0 for i in students}
    unassigned = set(students)

    while True:
        proposals = {s: [] for s in schools}
        any_proposal = False

        for i in list(unassigned):
            if next_choice[i] < len(schools):
                s = prefs[i][next_choice[i]]
                next_choice[i] += 1
                proposals[s].append(i)
                any_proposal = True
            else:
                unassigned.remove(i)

        if not any_proposal:
            break

        for s in schools:
            cand = held[s] + proposals[s]
            cand.sort(key=lambda st: pr_rank[s][st])  # lower index = higher priority
            held[s] = cand[:cap[s]]

        assigned = set(itertools.chain.from_iterable(held.values()))
        unassigned = set(students) - assigned

    matching = {i: None for i in students}
    for s, lst in held.items():
        for i in lst:
            matching[i] = s
    return matching

## IA (Boston): immediate acceptance

In [18]:
def immediate_acceptance(prefs, priorities, cap):
    pr_rank = {s: {stu: idx for idx, stu in enumerate(priorities[s])} for s in schools}
    remaining_cap = cap.copy()
    unassigned = set(students)
    matching = {i: None for i in students}

    # Round r: apply to r-th choice (r=0,1,2)
    for r in range(len(schools)):
        proposals = {s: [] for s in schools}
        for i in list(unassigned):
            s = prefs[i][r]
            proposals[s].append(i)

        for s, props in proposals.items():
            if remaining_cap[s] == 0 or not props:
                continue
            props.sort(key=lambda st: pr_rank[s][st])
            accept = props[:remaining_cap[s]]
            for i in accept:
                matching[i] = s
                unassigned.remove(i)
            remaining_cap[s] -= len(accept)

        if not unassigned:
            break

    return matching

## TTC with counters (school choice TTC)

In [19]:
def ttc_school_choice(prefs, priorities, cap):
    pr_rank = {s: {stu: idx for idx, stu in enumerate(priorities[s])} for s in schools}
    remaining_cap = cap.copy()
    remaining_students = set(students)
    matching = {i: None for i in students}

    while remaining_students:
        active_schools = [s for s in schools if remaining_cap[s] > 0]

        # students point to their top active school
        stu_to = {}
        for i in remaining_students:
            for s in prefs[i]:
                if remaining_cap[s] > 0:
                    stu_to[i] = s
                    break

        # schools point to best (highest-priority) student among those pointing to them
        sch_to = {}
        for s in active_schools:
            stus = [i for i, ss in stu_to.items() if ss == s]
            if stus:
                stus.sort(key=lambda i: pr_rank[s][i])
                sch_to[s] = stus[0]

        # find cycles in the directed graph
        cycles = []
        visited = set()

        for start in list(remaining_students):
            if start in visited:
                continue
            path = []
            idx = {}
            node = start

            while True:
                if node in idx:
                    cyc = path[idx[node]:]
                    cycles.append(cyc)
                    break

                idx[node] = len(path)
                path.append(node)
                visited.add(node)

                if node in remaining_students:
                    node = stu_to[node]
                else:
                    node = sch_to.get(node, None)

                if node is None:
                    break

        if not cycles:
            break

        # execute cycles: assign each student in cycle to their pointed school
        for cyc in cycles:
            cyc_students = [n for n in cyc if n in remaining_students]
            for i in cyc_students:
                s = stu_to[i]
                matching[i] = s

            for i in cyc_students:
                s = matching[i]
                remaining_students.remove(i)
                remaining_cap[s] -= 1

    return matching


## Generate output(latex)

In [20]:

def matching_to_latex_table(da, ia, ttc,
                            caption="One-draw matching outcomes (seed=42)",
                            label="tab:matching"):
    # oder :i1, i2, ..., i18 
    stus = sorted(da.keys(), key=lambda x: int(x[1:]))

    lines = []
    lines.append(r"\begin{table}[H]")
    lines.append(r"\centering")
    lines.append(r"\caption{" + caption + r"}")
    lines.append(r"\label{" + label + r"}")
    lines.append(r"\begin{tabular}{lccc}")
    lines.append(r"\hline")
    lines.append(r"Student & DA & IA & TTC \\")
    lines.append(r"\hline")

    for i in stus:
        lines.append(f"{i} & {da[i]} & {ia[i]} & {ttc[i]} \\\\")

    lines.append(r"\hline")
    lines.append(r"\end{tabular}")
    lines.append(r"\end{table}")

    return "\n".join(lines)

if __name__ == "__main__":
    # ---- One draw (Part 2 report) ----
    random.seed(42)
    prefs = random_prefs()
    prior = random_priorities()

    da  = deferred_acceptance(prefs, prior, cap0)
    ia  = immediate_acceptance(prefs, prior, cap0)
    ttc = ttc_school_choice(prefs, prior, cap0)

    # print outcomes
    print("One-draw matching outcomes (seed=42):")
    print("DA :", da)
    print("IA :", ia)
    print("TTC:", ttc)

    # ---- generate tex file ----
    tex = matching_to_latex_table(da, ia, ttc,
                                  caption="One-draw matching outcomes (seed=42)",
                                  label="tab:matching_seed42")
    out_path = "./table/table2_1.tex"
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(tex)

    print(f"\nSaved LaTeX table to: {out_path}")

One-draw matching outcomes (seed=42):
DA : {'i1': 's2', 'i2': 's2', 'i3': 's1', 'i4': 's3', 'i5': 's1', 'i6': 's1', 'i7': 's3', 'i8': 's1', 'i9': 's3', 'i10': 's2', 'i11': 's1', 'i12': 's1', 'i13': 's2', 'i14': 's2', 'i15': 's2', 'i16': 's3', 'i17': 's3', 'i18': 's3'}
IA : {'i1': 's2', 'i2': 's3', 'i3': 's1', 'i4': 's3', 'i5': 's1', 'i6': 's1', 'i7': 's3', 'i8': 's1', 'i9': 's2', 'i10': 's2', 'i11': 's1', 'i12': 's1', 'i13': 's2', 'i14': 's2', 'i15': 's2', 'i16': 's3', 'i17': 's3', 'i18': 's3'}
TTC: {'i1': 's2', 'i2': 's3', 'i3': 's1', 'i4': 's3', 'i5': 's1', 'i6': 's1', 'i7': 's3', 'i8': 's1', 'i9': 's2', 'i10': 's2', 'i11': 's1', 'i12': 's1', 'i13': 's2', 'i14': 's2', 'i15': 's2', 'i16': 's3', 'i17': 's3', 'i18': 's3'}

Saved LaTeX table to: ./table/table2_1.tex


# Part3: Effciency Analysis

In [21]:


def avg_rank(matching, prefs):
    """
    Average rank of assigned school across all students.
    Rank: 1 = best, 2 = second, 3 = worst.
    """
    ranks = rank_map(prefs)
    return sum(ranks[i][matching[i]] for i in students) / len(students)

def simulate(N=1000, seed=123):
    """
    Run N random markets, compute average assigned-school rank for each mechanism.
    Returns dict: {"DA": ..., "IA": ..., "TTC": ...}
    """
    random.seed(seed)
    sums = {"DA": 0.0, "IA": 0.0, "TTC": 0.0}

    for _ in range(N):
        prefs = random_prefs()
        prior = random_priorities()

        da  = deferred_acceptance(prefs, prior, cap0)
        ia  = immediate_acceptance(prefs, prior, cap0)
        ttc = ttc_school_choice(prefs, prior, cap0)

        sums["DA"]  += avg_rank(da, prefs)
        sums["IA"]  += avg_rank(ia, prefs)
        sums["TTC"] += avg_rank(ttc, prefs)

    return {k: v / N for k, v in sums.items()}

def avg_rank_results_to_latex_table(results,
                                   caption="Average assigned-school rank (N=1000, seed=123)",
                                   label="tab:avg_rank_seed123"):
    """
    Convert results dict into a LaTeX table string.
    """
    order = ["DA", "IA", "TTC"]

    lines = []
    lines.append(r"\begin{table}[H]")
    lines.append(r"\centering")
    lines.append(r"\caption{" + caption + r"}")
    lines.append(r"\label{" + label + r"}")
    lines.append(r"\begin{tabular}{lc}")
    lines.append(r"\hline")
    lines.append(r"Mechanism & Avg. rank \\")
    lines.append(r"\hline")

    for mech in order:
        lines.append(f"{mech} & {results[mech]:.4f} \\\\")

    lines.append(r"\hline")
    lines.append(r"\end{tabular}")
    lines.append(r"\end{table}")

    return "\n".join(lines)

# ---- Run Part 3 and save tex ----
N_SIM = 1000
SEED3 = 123

results = simulate(N=N_SIM, seed=SEED3)

print("\nAverage assigned-school rank (lower is better):")
for k, v in results.items():
    print(f"{k}: {v:.4f}")

tex_avg = avg_rank_results_to_latex_table(
    results,
    caption=f"Average assigned-school rank (N={N_SIM}, seed={SEED3})",
    label=f"tab:avg_rank_seed{SEED3}"
)

with open("./table/table3_1.tex", "w", encoding="utf-8") as f:
    f.write(tex_avg)

print("\nSaved LaTeX table to: table3_1.tex")


Average assigned-school rank (lower is better):
DA: 1.2142
IA: 1.1862
TTC: 1.1862

Saved LaTeX table to: table3_1.tex
